# Digital Witness — Full Pipeline
**YOLO26n + MobileNetV2 + Rolling Average + POS-Ready Product Counting**

### Architecture
```
Video Frame
    ↓
YOLO26n (fine-tuned on your 24-class dataset)
    ├── Person detection + ByteTrack ID
    ├── Product/basket/bag state per person
    └── Checkout counter state
    ↓
Person crop → MobileNetV2 → Rolling Average Queue
    └── Behaviour label: normal / shoplifting
    ↓
Product Counter (per tracked person)
    └── filled_items_seen, concealment_events, checkout_bypassed
    ↓
Intent Scorer + Bias Assessment (XAI)
    ↓
POS Comparator  ←── Transaction data (plug in later)
    └── products_detected vs products_billed → mismatch flag
    ↓
Case File + Alert
```

### Your 24 YOLO classes — how they map to the pipeline
```
PERSON CLASSES (tracking):  person, person-with-*, ...
PRODUCT CLASSES (counting): carrying-item, filled-basket, filled-shopping-bags,
                             filled-trolly, empty-basket, empty-shopping-bags,
                             empty-trolly, backpack-or-handbag
CHECKOUT CLASSES (POS):     occupied-checkout-counter, vacant-checkout-counter
CONCEALMENT SIGNAL:         person-with-backpack-handbag (items going into bag)
                             person-with-carrying-item (holding unpaid item)
```

### Dataset structure expected
```
YOLO dataset:   data/dataset/train/images + labels,  valid/images + labels
Behaviour vids: Dataset/normal/  +  Dataset/shoplifting/
```

In [10]:
# ============================================================
# CELL 1 — Install Dependencies (Local)
# ============================================================
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages, '--quiet'])

# ---- Detect CUDA version and install correct PyTorch ----
try:
    import torch
    if torch.cuda.is_available():
        print(f'PyTorch {torch.__version__} with CUDA already installed — skipping torch install.')
        _torch_ok = True
    else:
        print(f'PyTorch {torch.__version__} installed but CUDA not available — reinstalling with CUDA support.')
        _torch_ok = False
except ImportError:
    print('PyTorch not found — installing.')
    _torch_ok = False

if not _torch_ok:
    # Detect CUDA version from nvidia-smi
    try:
        out = subprocess.check_output(['nvidia-smi'], text=True)
        # Parse "CUDA Version: XX.X" from nvidia-smi output
        import re
        match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', out)
        cuda_major = int(match.group(1)) if match else 0
    except Exception:
        cuda_major = 0

    if cuda_major >= 12:
        index_url = 'https://download.pytorch.org/whl/cu121'
        print(f'Detected CUDA {cuda_major}.x — installing PyTorch with cu121 support...')
    elif cuda_major == 11:
        index_url = 'https://download.pytorch.org/whl/cu118'
        print(f'Detected CUDA {cuda_major}.x — installing PyTorch with cu118 support...')
    else:
        index_url = None
        print('No CUDA GPU detected — installing CPU-only PyTorch.')

    if index_url:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            'torch', 'torchvision',
            '--index-url', index_url, '--quiet'
        ])
    else:
        pip_install('torch', 'torchvision')

# ---- Install remaining packages ----
pip_install(
    'opencv-python',
    'numpy', 'pandas',
    'matplotlib',
    'scikit-learn',
    'Pillow',
    'tqdm',
    'ultralytics>=8.3.0',
    'lapx>=0.5.2',
    'pyyaml',
)

# ---- Verify ----
import torch
print(f'\nPyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}', end=' ')
if torch.cuda.is_available():
    print(f'({torch.cuda.get_device_name(0)})')
else:
    print('— running on CPU')
    print('\nIf you have an NVIDIA GPU and CUDA is still False:')
    print('  1. Run: nvidia-smi   (check your CUDA version)')
    print('  2. Manually install: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121')

PyTorch 2.10.0+cpu installed but CUDA not available — reinstalling with CUDA support.
No CUDA GPU detected — installing CPU-only PyTorch.

PyTorch  : 2.10.0+cpu
CUDA     : False — running on CPU

If you have an NVIDIA GPU and CUDA is still False:
  1. Run: nvidia-smi   (check your CUDA version)
  2. Manually install: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121


In [11]:
# ============================================================
# CELL 2 — Configuration (Local)
# ============================================================
import torch
import os, yaml, json
from pathlib import Path
from datetime import datetime

# ---- Paths ----
ROOT           = Path('C:/Users/MSI/Music/Project_DigitalWitness').resolve()                          # project root
VIDEO_ROOT     = Path('C:/Users/MSI/Music/Dataset')        # normal/ and shoplifting/ videos
FRAMES_DIR     = ROOT / 'frames'                             # extracted frames (local SSD)
DATASET_FOLDER = ROOT / 'data' / 'dataset'                   # YOLO annotation dataset (data.yaml here)
MODELS_DIR     = ROOT / 'models'
OUTPUTS_DIR    = ROOT / 'outputs'
CASE_DIR       = OUTPUTS_DIR / 'cases'

for d in [MODELS_DIR, OUTPUTS_DIR, CASE_DIR, FRAMES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device         : {device}')
print(f'Root           : {ROOT}')
print(f'Video root     : {VIDEO_ROOT}  (exists: {VIDEO_ROOT.exists()})')
print(f'Dataset folder : {DATASET_FOLDER}  (exists: {DATASET_FOLDER.exists()})')

# ---- Locate and patch data.yaml ----
DATASET_YAML = str(DATASET_FOLDER / 'data.yaml')

if Path(DATASET_YAML).exists():
    with open(DATASET_YAML) as f:
        _y = yaml.safe_load(f)

    _y['train'] = str(DATASET_FOLDER / 'train' / 'images')
    _y['val']   = str(DATASET_FOLDER / 'valid' / 'images')
    _y['test']  = str(DATASET_FOLDER / 'test'  / 'images')
    _y.pop('path', None)

    with open(DATASET_YAML, 'w') as f:
        yaml.dump(_y, f, sort_keys=False, allow_unicode=True)

    print(f'\ndata.yaml patched with absolute paths')
    for split, p in [('train', _y['train']), ('val', _y['val'])]:
        exists = Path(p).exists()
        print(f'  {split} → {p}  {"OK" if exists else "WARNING: folder not found"}')
else:
    print(f'\nWARNING: data.yaml not found at {DATASET_YAML}')
    print('Place your Roboflow YOLOv8 export in data/dataset/')

# ---- Your 24 YOLO classes ----
YOLO_CLASSES = [
    'backpack-or-handbag',                                    # 0
    'carrying-item',                                          # 1
    'empty-basket',                                           # 2
    'empty-shopping-bags',                                    # 3
    'empty-trolly',                                           # 4
    'filled-basket',                                          # 5
    'filled-shopping-bags',                                   # 6
    'filled-trolly',                                          # 7
    'occupied-checkout-counter',                              # 8
    'person',                                                 # 9
    'person with carrying item-s- and shopping bag-s-',       # 10
    'person-with-backpack-handbag',                           # 11
    'person-with-backpack-handbag-and-carrying-item-s-',      # 12
    'person-with-backpack-handbag-and-shopping-bag-s-',       # 13
    'person-with-basket-trolly-and-backpack-handbag',         # 14
    'person-with-basket-trolly-and-carrying-item',            # 15
    'person-with-basket-trolly-and-shopping-bag',             # 16
    'person-with-basket-trolly-shopping-bag-s-and-backpack-handbag', # 17
    'person-with-carrying-item',                              # 18
    'person-with-empty-basket-trolly',                        # 19
    'person-with-empty-shopping-bag-s-',                      # 20
    'person-with-filled-basket-trolly',                       # 21
    'person-with-filled-shopping-bag-s-',                     # 22
    'vacant-checkout-counter',                                # 23
]

# ---- Class groupings ----
PERSON_CLASS_IDS = {9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22}

PRODUCT_HELD_IDS = {
    1,   # carrying-item
    5,   # filled-basket
    6,   # filled-shopping-bags
    7,   # filled-trolly
    10,  # person with carrying item-s- and shopping bag-s-
    15,  # person-with-basket-trolly-and-carrying-item
    16,  # person-with-basket-trolly-and-shopping-bag
    18,  # person-with-carrying-item
    21,  # person-with-filled-basket-trolly
    22,  # person-with-filled-shopping-bag-s-
}

CONCEALMENT_IDS = {
    0,   # backpack-or-handbag (standalone)
    11,  # person-with-backpack-handbag
    12,  # person-with-backpack-handbag-and-carrying-item-s-
    13,  # person-with-backpack-handbag-and-shopping-bag-s-
    14,  # person-with-basket-trolly-and-backpack-handbag
    17,  # person-with-basket-trolly-shopping-bag-s-and-backpack-handbag
}

CHECKOUT_OCCUPIED_ID = 8
CHECKOUT_VACANT_ID   = 23

# ---- MobileNetV2 behaviour classes ----
BEHAVIOR_CLASSES = ['normal', 'shoplifting']

# ---- Model save paths ----
YOLO26_BASE    = str(MODELS_DIR / 'yolo26n.pt')
YOLO26_RETAIL  = str(MODELS_DIR / 'yolo26_retail.pt')
MOBILENET_SAVE = str(MODELS_DIR / 'mobilenet_dw.pt')
MOBILENET_INFO = str(MODELS_DIR / 'mobilenet_dw_info.json')

# ---- YOLO inference settings ----
YOLO_CONF = 0.35
YOLO_IOU  = 0.45

# ---- MobileNetV2 settings ----
FRAME_SIZE    = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---- Training hyperparameters ----
# NOTE: Set to 2 for quick test. Change to EPOCHS_YOLO=50, EPOCHS_MOBILENET=25 for real training.
EPOCHS_YOLO      = 2
EPOCHS_MOBILENET = 2
BATCH_SIZE       = 32
LR_MOBILENET     = 1e-4
PATIENCE         = 5

# ---- Intent scoring thresholds ----
THRESHOLD_LOW      = 0.30
THRESHOLD_MEDIUM   = 0.50
THRESHOLD_HIGH     = 0.70
THRESHOLD_CRITICAL = 0.85

QUEUE_SIZE = 30   # rolling average window — ~1 sec at 30fps

print(f'\nYOLO classes     : {len(YOLO_CLASSES)}')
print(f'Behaviour cls    : {BEHAVIOR_CLASSES}')
print(f'EPOCHS_YOLO      : {EPOCHS_YOLO}')
print(f'EPOCHS_MOBILENET : {EPOCHS_MOBILENET}')

Device         : cpu
Root           : C:\Users\MSI\Music\Project_DigitalWitness
Video root     : C:\Users\MSI\Music\Dataset  (exists: True)
Dataset folder : C:\Users\MSI\Music\Project_DigitalWitness\data\dataset  (exists: True)

data.yaml patched with absolute paths
  train → C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\train\images  OK
  val → C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\valid\images  OK

YOLO classes     : 24
Behaviour cls    : ['normal', 'shoplifting']
EPOCHS_YOLO      : 2
EPOCHS_MOBILENET : 2


In [ ]:
# ============================================================
# CELL 3 — YOLO26n Fine-tuning
# ============================================================
from ultralytics import YOLO
import shutil

if not Path(YOLO26_BASE).exists():
    raise FileNotFoundError(
        f'YOLO26n base weights not found at {YOLO26_BASE}\n'
        'Download yolo26n.pt and place it in the models/ folder.'
    )

if not Path(DATASET_YAML).exists():
    raise FileNotFoundError(f'data.yaml not found at {DATASET_YAML}')

print(f'Base weights : {YOLO26_BASE}')
print(f'Dataset YAML : {DATASET_YAML}')
print(f'Training for : {EPOCHS_YOLO} epochs, batch={BATCH_SIZE}, freeze=10')

yolo = YOLO(YOLO26_BASE)

results = yolo.train(
    data    = DATASET_YAML,
    epochs  = EPOCHS_YOLO,
    imgsz   = 640,
    batch   = BATCH_SIZE,
    freeze  = 10,
    project = str(ROOT / 'runs'),
    name    = 'yolo26_retail',
    save    = True,
    patience= 5,
    plots   = True,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    verbose = True,
)

# Use results.save_dir — works regardless of auto-incremented folder name
weights_dir = Path(results.save_dir) / 'weights'
best = weights_dir / 'best.pt'
last = weights_dir / 'last.pt'

src = best if best.exists() else last if last.exists() else None
if src is None:
    raise FileNotFoundError(
        f'No weights found in {weights_dir}\n'
        f'Check that training completed successfully.'
    )

shutil.copy(str(src), YOLO26_RETAIL)
print(f'\nFine-tuned YOLO saved → {YOLO26_RETAIL}  (from {src.name})')

try:
    m = results.results_dict
    map50 = m.get('metrics/mAP50(B)', None)
    if map50 is None:
        # Key mismatch — print actual keys so you can see what Ultralytics returned
        print(f'WARNING: metric key not found. Available keys: {list(m.keys())}')
        map50 = 0.0
    print(f"mAP50     : {map50:.3f}")
    print(f"mAP50-95  : {m.get('metrics/mAP50-95(B)', 0):.3f}")
    print(f"Precision : {m.get('metrics/precision(B)', 0):.3f}")
    print(f"Recall    : {m.get('metrics/recall(B)', 0):.3f}")
    if EPOCHS_YOLO < 5:
        print(f'\nNOTE: Metrics are expected to be ~0 with only {EPOCHS_YOLO} epoch(s). '
              f'Set EPOCHS_YOLO=50 for real training.')
except Exception as e:
    print(f'(metrics parse error: {e})')

Base weights : C:\Users\MSI\Music\Project_DigitalWitness\models\yolo26n.pt
Dataset YAML : C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\data.yaml
Training for : 2 epochs, batch=32, freeze=10
New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.11  Python-3.12.2 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, h

In [ ]:
# ============================================================
# CELL 4 — MobileNetV2 Training (behaviour classifier)
# ============================================================
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random
from PIL import Image
from tqdm import tqdm

# ---- Step 4a: Extract frames from behaviour videos ----
def extract_frames(video_root, output_dir, classes, fps_target=2):
    total = 0
    for cls in classes:
        src = Path(video_root) / cls
        dst = Path(output_dir) / cls
        dst.mkdir(parents=True, exist_ok=True)

        existing = len(list(dst.glob('*.jpg')))
        if existing > 0:
            print(f'[{cls}] {existing} frames already extracted — skipping')
            total += existing
            continue

        videos = (list(src.glob('*.mp4')) + list(src.glob('*.avi'))
                  + list(src.glob('*.mov')))
        if not videos:
            print(f'WARNING: No videos found in {src}')
            continue

        print(f'[{cls}] Extracting from {len(videos)} videos...')
        count = 0
        for vid in tqdm(videos, desc=cls):
            cap      = cv2.VideoCapture(str(vid))
            vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 25
            interval = max(1, int(vid_fps / fps_target))
            fi = 0
            while True:
                ret, frame = cap.read()
                if not ret: break
                if fi % interval == 0:
                    out = dst / f'{vid.stem}_f{fi:06d}.jpg'
                    cv2.imwrite(str(out), cv2.resize(frame, FRAME_SIZE))
                    count += 1
                fi += 1
            cap.release()
        print(f'[{cls}] {count} frames extracted')
        total += count
    return total

print('Extracting frames from behaviour videos...')
n = extract_frames(VIDEO_ROOT, FRAMES_DIR, BEHAVIOR_CLASSES, fps_target=2)
print(f'Total: {n} frames')
for cls in BEHAVIOR_CLASSES:
    k = len(list((Path(FRAMES_DIR) / cls).glob('*.jpg')))
    print(f'  {cls}: {k}')

# ---- Step 4b: Dataset and DataLoaders ----
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FrameDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

all_samples = []
for i, cls in enumerate(BEHAVIOR_CLASSES):
    for p in (Path(FRAMES_DIR) / cls).glob('*.jpg'):
        all_samples.append((str(p), i))
random.shuffle(all_samples)

labels_all = [l for _, l in all_samples]
train_s, val_s = train_test_split(all_samples, test_size=0.2,
                                   stratify=labels_all, random_state=42)
print(f'\nTrain: {len(train_s)}  |  Val: {len(val_s)}')

train_dl = DataLoader(FrameDataset(train_s, train_tf), batch_size=BATCH_SIZE,
                      shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(FrameDataset(val_s,   val_tf),   batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=2, pin_memory=True)

# ---- Step 4c: Build MobileNetV2 ----
base = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
for p in base.parameters(): p.requires_grad = False       # freeze all
for layer in base.features[16:]:                          # unfreeze last 3 blocks
    for p in layer.parameters(): p.requires_grad = True
base.classifier = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(1280, 512),
    nn.ReLU(),       nn.Dropout(0.3),
    nn.Linear(512, len(BEHAVIOR_CLASSES)),
)
mobilenet = base.to(device)
trainable = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

# ---- Step 4d: Training loop ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, mobilenet.parameters()),
                       lr=LR_MOBILENET)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc, no_improve = 0.0, 0

for epoch in range(1, EPOCHS_MOBILENET + 1):
    mobilenet.train()
    tl, tc, tt = 0, 0, 0
    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(mobilenet(imgs), lbls)
        loss.backward(); optimizer.step()
        tl += loss.item() * len(imgs)
        tc += (mobilenet(imgs).argmax(1) == lbls).sum().item()
        tt += len(imgs)

    mobilenet.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = mobilenet(imgs)
            vl += criterion(out, lbls).item() * len(imgs)
            vc += (out.argmax(1) == lbls).sum().item()
            vt += len(imgs)

    tl /= tt; vl /= vt; ta = tc/tt; va = vc/vt
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)

    # Log LR for visibility (replaces verbose=True)
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{EPOCHS_MOBILENET}  '
          f'train_loss={tl:.4f}  train_acc={ta:.3f}  '
          f'val_loss={vl:.4f}  val_acc={va:.3f}  lr={current_lr:.2e}')
    scheduler.step(vl)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(mobilenet.state_dict(), MOBILENET_SAVE)
        no_improve = 0
        print(f'  ✓ Best val_acc={va:.3f} saved')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

# Save model info
with open(MOBILENET_INFO, 'w') as f:
    json.dump({'classes': BEHAVIOR_CLASSES, 'best_val_acc': best_val_acc,
               'input_size': list(FRAME_SIZE),
               'mean': IMAGENET_MEAN, 'std': IMAGENET_STD}, f, indent=2)
print(f'\nBest val accuracy: {best_val_acc:.3f}')
print(f'Model saved → {MOBILENET_SAVE}')

In [ ]:
# ============================================================
# CELL 5 — Evaluation (MobileNetV2)
# ============================================================
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
import numpy as np

mobilenet.load_state_dict(torch.load(MOBILENET_SAVE, map_location=device))
mobilenet.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_dl:
        preds = mobilenet(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(lbls.numpy())

print('=== MobileNetV2 Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=BEHAVIOR_CLASSES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=BEHAVIOR_CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Digital Witness')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'learning_curve.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 6 — Product Tracker + POS Integration
# ============================================================
# Imports pos_integration.py which handles:
#   - MockPOSDatabase  : generates realistic test transactions
#   - POSComparator    : compares YOLO product counts vs billed
#   - prompt_operator_verification : interactive operator prompt
#   - generate_pos_report : saves audit trail
#
# On Colab: make sure pos_integration.py is in your dev/ folder
# on Google Drive, then the sys.path line below will find it.
# ============================================================
import sys

# Add the dev/ folder to path so we can import pos_integration
if ON_COLAB:
    sys.path.insert(0, str(ROOT))   # ROOT = /content/drive/MyDrive/DigitalWithness/dev
else:
    sys.path.insert(0, str(ROOT))

from pos_integration import (
    MockPOSDatabase,
    POSComparator,
    prompt_operator_verification,
    generate_pos_report,
)

# ---- PersonProductTracker (unchanged — still needed by inference) ----
class PersonProductTracker:
    """Tracks product state for a single tracked person (one ByteTrack ID)."""
    def __init__(self, track_id):
        self.track_id           = track_id
        self.max_products_held  = 0
        self.concealment_frames = 0
        self.checkout_visited   = False
        self.product_frames     = 0
        self.total_frames       = 0
        self.class_history      = []

    def update(self, detected_class_ids, checkout_occupied_nearby):
        self.total_frames += 1
        self.class_history.extend(list(detected_class_ids))
        products_now = len(detected_class_ids & PRODUCT_HELD_IDS)
        self.max_products_held = max(self.max_products_held, products_now)
        if products_now > 0:
            self.product_frames += 1
        if detected_class_ids & CONCEALMENT_IDS:
            self.concealment_frames += 1
        if checkout_occupied_nearby:
            self.checkout_visited = True

    @property
    def left_with_goods(self):
        return self.max_products_held > 0 and not self.checkout_visited

    @property
    def concealment_ratio(self):
        return self.concealment_frames / max(1, self.total_frames)

    def to_dict(self):
        return {
            'track_id'          : self.track_id,
            'max_products_held' : self.max_products_held,
            'concealment_frames': self.concealment_frames,
            'concealment_ratio' : round(self.concealment_ratio, 3),
            'checkout_visited'  : self.checkout_visited,
            'left_with_goods'   : self.left_with_goods,
            'total_frames'      : self.total_frames,
        }

# ---- Generate mock POS database for testing ----
POS_DB_PATH = str(OUTPUTS_DIR / 'mock_pos_transactions.json')

pos_db = MockPOSDatabase()
pos_db.generate_sessions(n=20)

# Add a suspicious session at the current time for easy testing
suspicious_txn = pos_db.add_suspicious_session(
    timestamp      = datetime.now(),
    items_billed   = 2,
    items_detected = 5,
)

pos_db.save(POS_DB_PATH)
pos_db.summary()

print()
print("POS database ready.")
print(f"Suspicious test session: {suspicious_txn['session_id']}")
print(f"  Billed: {suspicious_txn['items_billed']} items")
print(f"  (YOLO should detect ~5 items in the test video)")


In [ ]:
# ============================================================
# CELL 7 — Full Inference Pipeline
# ============================================================
# YOLO26n detects persons + products per frame.
# For each tracked person:
#   - Crop person region → MobileNetV2 → rolling average
#   - ProductTracker updated from YOLO class IDs
# Outputs: behaviour events + per-person product counts
# ============================================================
import torch.nn.functional as F
from collections import deque, defaultdict
import cv2
import numpy as np
from torchvision import transforms, models
import torch.nn as nn

# Inference transform for MobileNetV2
inf_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FRAME_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_mobilenet(model_path):
    m = models.mobilenet_v2(weights=None)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(1280, 512),
        nn.ReLU(),       nn.Dropout(0.3),
        nn.Linear(512, len(BEHAVIOR_CLASSES)),
    )
    m.load_state_dict(torch.load(model_path, map_location=device))
    return m.to(device).eval()


def run_inference(video_path,
                  yolo_path     = YOLO26_RETAIL,
                  mobilenet_path= MOBILENET_SAVE,
                  queue_size    = QUEUE_SIZE,
                  frame_step    = 1):
    """
    Full inference on a video.

    Returns dict with:
        overall_class       — 'normal' or 'shoplifting'
        overall_conf        — confidence 0-1
        behavior_events     — list of timed behaviour segments
        person_summaries    — per-tracked-person product counts (POS-ready)
        duration, fps, total_frames
    """
    from ultralytics import YOLO

    yolo_model = YOLO(yolo_path)
    mn_model   = load_mobilenet(mobilenet_path)

    cap          = cv2.VideoCapture(str(video_path))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration     = total_frames / fps

    # Per-person rolling queues and product trackers
    person_queues   = defaultdict(lambda: deque(maxlen=queue_size))
    person_trackers = {}   # track_id → PersonProductTracker

    behavior_events = []
    current_event   = None
    frame_num       = 0

    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame_num += 1
            if frame_num % frame_step != 0: continue

            timestamp = frame_num / fps

            # ---- YOLO detection + ByteTrack ----
            yolo_results = yolo_model.track(
                frame, persist=True,
                conf=YOLO_CONF, iou=YOLO_IOU, verbose=False
            )[0]

            # Check if any checkout counter is occupied this frame
            frame_classes = set()
            if yolo_results.boxes is not None and len(yolo_results.boxes):
                for cls_id in yolo_results.boxes.cls.cpu().numpy():
                    frame_classes.add(int(cls_id))
            checkout_occupied = CHECKOUT_OCCUPIED_ID in frame_classes

            # ---- Per-person processing ----
            frame_pred_label = 'normal'
            frame_pred_conf  = 0.5

            if yolo_results.boxes is not None and len(yolo_results.boxes):
                boxes   = yolo_results.boxes
                cls_ids = boxes.cls.cpu().numpy().astype(int)
                confs   = boxes.conf.cpu().numpy()
                xyxy    = boxes.xyxy.cpu().numpy().astype(int)

                # Get track IDs (ByteTrack) — may be None if tracking not active
                track_ids = (boxes.id.cpu().numpy().astype(int)
                             if boxes.id is not None
                             else np.arange(len(boxes)))

                # Group class detections by track_id
                person_classes = defaultdict(set)
                person_boxes   = {}

                for i, (cls_id, tid) in enumerate(zip(cls_ids, track_ids)):
                    if cls_id in PERSON_CLASS_IDS or cls_id == 9:
                        person_classes[tid].add(cls_id)
                        person_boxes[tid] = xyxy[i]   # last box wins

                # Process each tracked person
                for tid, cls_set in person_classes.items():
                    # Initialise tracker if new
                    if tid not in person_trackers:
                        person_trackers[tid] = PersonProductTracker(tid)

                    # Update product tracker
                    person_trackers[tid].update(cls_set, checkout_occupied)

                    # Crop person region for MobileNetV2
                    x1, y1, x2, y2 = person_boxes[tid]
                    x1, y1 = max(0, x1), max(0, y1)
                    x2 = min(frame.shape[1], x2)
                    y2 = min(frame.shape[0], y2)

                    if x2 > x1 and y2 > y1:
                        crop  = frame[y1:y2, x1:x2]
                        rgb   = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                        tensor= inf_tf(rgb).unsqueeze(0).to(device)

                        logits = mn_model(tensor)
                        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]
                        person_queues[tid].append(probs)

                        # Rolling average for this person
                        avg = np.array(person_queues[tid]).mean(axis=0)
                        pid = int(avg.argmax())

                        # Use highest-risk person's prediction for frame label
                        if BEHAVIOR_CLASSES[pid] == 'shoplifting' and avg[pid] > frame_pred_conf:
                            frame_pred_label = 'shoplifting'
                            frame_pred_conf  = float(avg[pid])
                        elif frame_pred_label == 'normal':
                            frame_pred_conf = float(avg[pid])

            # ---- Build behaviour event segments ----
            if current_event is None:
                current_event = {
                    'behavior_type': frame_pred_label,
                    'start_time'   : timestamp,
                    'end_time'     : timestamp,
                    'confidence'   : frame_pred_conf,
                    'probabilities': {'normal': 1 - frame_pred_conf
                                      if frame_pred_label == 'shoplifting'
                                      else frame_pred_conf,
                                      'shoplifting': frame_pred_conf
                                      if frame_pred_label == 'shoplifting'
                                      else 1 - frame_pred_conf},
                }
            elif frame_pred_label == current_event['behavior_type']:
                current_event['end_time']   = timestamp
                current_event['confidence'] = max(current_event['confidence'],
                                                   frame_pred_conf)
            else:
                behavior_events.append(current_event)
                current_event = {
                    'behavior_type': frame_pred_label,
                    'start_time'   : timestamp,
                    'end_time'     : timestamp,
                    'confidence'   : frame_pred_conf,
                    'probabilities': {'normal': 1 - frame_pred_conf
                                      if frame_pred_label == 'shoplifting'
                                      else frame_pred_conf,
                                      'shoplifting': frame_pred_conf
                                      if frame_pred_label == 'shoplifting'
                                      else 1 - frame_pred_conf},
                }

    cap.release()
    if current_event:
        behavior_events.append(current_event)

    # Overall classification (duration-weighted)
    shop_w = sum(e['confidence'] * (e['end_time'] - e['start_time'])
                 for e in behavior_events if e['behavior_type'] == 'shoplifting')
    norm_w = sum(e['confidence'] * (e['end_time'] - e['start_time'])
                 for e in behavior_events if e['behavior_type'] == 'normal')
    overall_class = 'shoplifting' if shop_w >= norm_w else 'normal'
    overall_conf  = min(1.0, (shop_w if overall_class == 'shoplifting'
                               else norm_w) / max(duration, 1))

    return {
        'overall_class'   : overall_class,
        'overall_conf'    : overall_conf,
        'is_shoplifting'  : overall_class == 'shoplifting',
        'behavior_events' : behavior_events,
        'person_summaries': {tid: t.to_dict()
                              for tid, t in person_trackers.items()},
        'duration'        : duration,
        'fps'             : fps,
        'total_frames'    : frame_num,
    }

print('run_inference() ready.')
print("Usage: result = run_inference('path/to/video.mp4')")

In [ ]:
# ============================================================
# CELL 8 — Intent Scoring + Bias-Aware Assessment
# ============================================================
# XAI layer — your thesis contribution.
# Now incorporates product count signals from YOLO.
# ============================================================

def calculate_intent_score(behavior_events, video_duration, person_summaries=None):
    """
    Compute 0-1 intent score from behaviour + product signals.

    Components:
      behaviour (MobileNetV2)  : 50%  — shoplifting classification signal
      product / concealment    : 30%  — YOLO product + concealment evidence
      duration                 : 20%  — time in suspicious state
    """
    shop_events = [e for e in behavior_events if e['behavior_type'] == 'shoplifting']

    # Behaviour score
    if shop_events:
        avg_conf = np.mean([e['confidence'] for e in shop_events])
        count_f  = min(1.0, len(shop_events) / 3.0)
        s_behav  = avg_conf * count_f
    else:
        s_behav = 0.0

    # Duration score
    sus_secs   = sum(e['end_time'] - e['start_time'] for e in shop_events)
    s_duration = min(1.0, (sus_secs / max(1.0, video_duration)) * 3.0)

    # Product / concealment score (from YOLO)
    s_product = 0.0
    product_flags = []
    if person_summaries:
        for tid, p in person_summaries.items():
            if p['left_with_goods']:
                s_product = max(s_product, 0.8)
                product_flags.append(f'Person {tid}: left store with goods, no checkout visit')
            if p['concealment_ratio'] > 0.3:
                s_product = max(s_product, p['concealment_ratio'])
                product_flags.append(f'Person {tid}: backpack/bag present {p["concealment_ratio"]:.0%} of time')
            if p['max_products_held'] > 0:
                product_flags.append(f'Person {tid}: max {p["max_products_held"]} product(s) detected')

    # Weighted sum
    total = 0.50 * s_behav + 0.30 * s_product + 0.20 * s_duration
    total = max(0.0, min(1.0, total))

    if total < THRESHOLD_LOW:        severity = 'NONE'
    elif total < THRESHOLD_MEDIUM:   severity = 'LOW'
    elif total < THRESHOLD_HIGH:     severity = 'MEDIUM'
    elif total < THRESHOLD_CRITICAL: severity = 'HIGH'
    else:                             severity = 'CRITICAL'

    explanation = []
    if shop_events:
        explanation.append(f'{len(shop_events)} shoplifting segment(s) ({sus_secs:.1f}s)')
    explanation.extend(product_flags)
    if not explanation:
        explanation.append('No suspicious activity detected')

    return {
        'score'      : total,
        'severity'   : severity,
        'explanation': '; '.join(explanation) + f'. Score: {total:.2f} ({severity})',
        'components' : {'behaviour': s_behav, 'product': s_product,
                         'duration': s_duration},
    }


def bias_aware_adjustment(intent_dict, quality_score=1.0):
    raw        = intent_dict['score']
    flags      = []
    adj        = 1.0
    if quality_score < 0.5:
        adj = min(adj, 0.70); flags.append('Low video quality')
    elif quality_score < 0.75:
        adj = min(adj, 0.85); flags.append('Moderate video quality')
    if raw >= 0.5: adj = adj ** 0.5
    adj_score = max(0.0, min(1.0, raw * adj))
    return {
        'raw_score'      : raw,
        'adjusted_score' : adj_score,
        'fairness_score' : max(0.0, quality_score * (1.0 - 0.2 * len(flags))),
        'adj_factor'     : adj,
        'flags'          : flags,
        'requires_review': len(flags) > 0 or raw > THRESHOLD_HIGH,
    }


def generate_alert(intent_dict, bias_result, behavior_events, pos_results=None):
    score    = bias_result['adjusted_score']
    severity = intent_dict['severity']
    if score < THRESHOLD_MEDIUM: return None

    n_sus    = sum(1 for e in behavior_events if e['behavior_type'] == 'shoplifting')
    alert_id = f"ALERT-{datetime.now().strftime('%Y%m%d%H%M%S')}"

    # Include POS mismatch in alert if available
    pos_note = ''
    if pos_results:
        mismatches = [r for r in pos_results.values()
                      if r.get('mismatch') is True]
        if mismatches:
            pos_note = f' POS MISMATCH: {len(mismatches)} person(s) with unaccounted items.'

    return {
        'alert_id'             : alert_id,
        'timestamp'            : datetime.now().isoformat(),
        'level'                : severity,
        'score'                : score,
        'message'              : (f'{n_sus} suspicious segment(s). '
                                   f'Risk: {score:.2f} ({severity}).{pos_note} '
                                   f'Human review required.'),
        'requires_human_review': True,
        'fairness_score'       : bias_result['fairness_score'],
    }


print('Intent scoring + bias assessment ready.')

In [ ]:
# ============================================================
# CELL 9 — End-to-End Analysis + Case File
# ============================================================
import matplotlib.patches as mpatches

def plot_behavior_timeline(behavior_events, duration, title='Behavior Timeline'):
    colors = {'normal': '#2ecc71', 'shoplifting': '#e74c3c'}
    fig, ax = plt.subplots(figsize=(14, 3))
    for e in behavior_events:
        ax.barh(0, max(0.1, e['end_time'] - e['start_time']),
                left=e['start_time'], height=0.6,
                color=colors.get(e['behavior_type'], '#95a5a6'),
                alpha=0.8, edgecolor='white')
    patches = [mpatches.Patch(color=c, label=k) for k, c in colors.items()
               if any(e['behavior_type'] == k for e in behavior_events)]
    ax.legend(handles=patches, loc='upper right', fontsize=9)
    ax.set_xlim(0, max(duration, 1))
    ax.set_xlabel('Time (seconds)'); ax.set_yticks([])
    ax.set_title(title, fontweight='bold')
    plt.tight_layout(); plt.show()


def analyze_video(video_path, quality_score=1.0, frame_step=1,
                  transaction=None):
    """
    Full end-to-end pipeline.

    Args:
        video_path    : path to video
        quality_score : 0-1 video quality estimate
        frame_step    : 1=all frames, 3=faster
        transaction   : dict with 'items_billed' for POS check
                        e.g. {'items_billed': 3}
                        Leave as None until POS is integrated.
    """
    print(f'Analyzing: {video_path}')

    # Step 1: Inference
    inf = run_inference(video_path, frame_step=frame_step)

    # Step 2: POS comparison (per tracked person)
    pos_results = {
        tid: compare_with_pos(PersonProductTracker.__new__(PersonProductTracker),
                              transaction)
        for tid, summary in inf['person_summaries'].items()
    }
    # Rebuild properly using actual tracker data
    pos_results = {}
    for tid, summary in inf['person_summaries'].items():
        t = PersonProductTracker(tid)
        t.max_products_held  = summary['max_products_held']
        t.concealment_frames = summary['concealment_frames']
        t.checkout_visited   = summary['checkout_visited']
        t.total_frames       = summary['total_frames']
        pos_results[tid]     = compare_with_pos(t, transaction)

    # Step 3: Intent scoring
    intent = calculate_intent_score(
        inf['behavior_events'], inf['duration'], inf['person_summaries'])

    # Step 4: Bias-aware adjustment
    bias = bias_aware_adjustment(intent, quality_score)

    # Step 5: Alert
    alert = generate_alert(intent, bias, inf['behavior_events'], pos_results)

    # Step 6: Print results
    sep = '=' * 65
    print(sep)
    print('  DIGITAL WITNESS — ANALYSIS RESULTS')
    print(sep)
    print(f'  Classification  : {inf["overall_class"].upper()}')
    print(f'  Confidence      : {inf["overall_conf"]:.1%}')
    print(f'  Intent Score    : {intent["score"]:.3f}  [{intent["severity"]}]')
    print(f'  Adjusted Score  : {bias["adjusted_score"]:.3f}')
    print(f'  Fairness Score  : {bias["fairness_score"]:.1%}')
    print(f'  Explanation     : {intent["explanation"]}')

    print(f'\n  --- Product Summary (per person) ---')
    for tid, p in inf['person_summaries'].items():
        pos = pos_results.get(tid, {})
        print(f'  Person {tid}: max_products={p["max_products_held"]}  '
              f'concealment={p["concealment_ratio"]:.0%}  '
              f'checkout={p["checkout_visited"]}  '
              f'left_with_goods={p["left_with_goods"]}')
        print(f'    POS: {pos.get("note", "N/A")}')

    if alert:
        print(f'\n  *** ALERT {alert["alert_id"]} [{alert["level"]}] ***')
        print(f'  {alert["message"]}')
    print(f'\n  NOTE: Advisory only. Human review required.')
    print(sep)

    # Step 7: Case file
    case_id   = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    case_data = {
        'case_id'         : case_id,
        'timestamp'       : datetime.now().isoformat(),
        'system'          : 'Digital Witness — YOLO26n + MobileNetV2 + Rolling Average',
        'video'           : {'path': str(video_path), 'duration': inf['duration'],
                              'fps': inf['fps'], 'frames': inf['total_frames']},
        'classification'  : {'label': inf['overall_class'],
                              'confidence': inf['overall_conf'],
                              'is_shoplifting': inf['is_shoplifting']},
        'behavior_events' : inf['behavior_events'],
        'person_summaries': inf['person_summaries'],
        'pos_results'     : pos_results,
        'intent_score'    : intent,
        'bias_assessment' : bias,
        'alert'           : alert,
        'advisory_note'   : 'Advisory system only. Does not determine guilt.',
    }
    case_path = CASE_DIR / f'{case_id}.json'
    with open(case_path, 'w') as f:
        json.dump(case_data, f, indent=2)

    if inf['behavior_events']:
        plot_behavior_timeline(
            inf['behavior_events'], inf['duration'],
            title=f'Behaviour Timeline — {Path(video_path).name}')

    print(f'Case file: {case_path}')
    return case_id, str(case_path), case_data


print('analyze_video() ready.')
print()
print("Basic:  case_id, path, r = analyze_video('video.mp4')")
print("Fast:   case_id, path, r = analyze_video('video.mp4', frame_step=3)")
print("+ POS:  case_id, path, r = analyze_video('video.mp4', transaction={'items_billed': 3})")

In [ ]:
# ============================================================
# CELL 10 — End-to-End Analysis with POS Verification
# ============================================================
# Full pipeline including the operator verification prompt.
#
# For testing: the mock database has a suspicious session
# timestamped to ~now. Run this cell with a test video and
# when prompted, try entering a number different from what
# YOLO detected to see the mismatch flag fire.
# ============================================================

def analyze_video_with_pos(video_path, quality_score=1.0, frame_step=1,
                             video_timestamp=None, tolerance_seconds=120):
    """
    Full end-to-end pipeline including POS operator verification.

    Args:
        video_path         : path to video file
        quality_score      : 0-1 video quality estimate (1.0 = good)
        frame_step         : process every Nth frame (1=all, 3=faster)
        video_timestamp    : datetime the video was recorded
                             (used to match POS transaction by time)
                             Defaults to now if not provided.
        tolerance_seconds  : how close video_timestamp must be to a
                             transaction timestamp to count as a match
    """
    if video_timestamp is None:
        video_timestamp = datetime.now()

    print(f'Analyzing: {video_path}')
    print(f'Video timestamp: {video_timestamp.strftime("%Y-%m-%d %H:%M:%S")}')
    print()

    # Step 1: Run full video inference (YOLO + MobileNetV2)
    inf = run_inference(video_path, frame_step=frame_step)

    # Step 2: Intent scoring (behaviour + product signals)
    intent = calculate_intent_score(
        inf['behavior_events'], inf['duration'], inf['person_summaries'])

    # Step 3: Bias-aware adjustment
    bias = bias_aware_adjustment(intent, quality_score)

    # Step 4: Interactive operator POS verification
    # This shows the operator what YOLO detected and prompts
    # them to confirm or enter the billed item count
    print("\n--- Starting Operator Verification ---\n")
    pos_results = prompt_operator_verification(
        inference_result  = inf,
        pos_db            = pos_db,
        video_timestamp   = video_timestamp,
        tolerance_seconds = tolerance_seconds,
    )

    # Step 5: Generate alert (now includes POS mismatch signals)
    alert = generate_alert(intent, bias, inf['behavior_events'], 
                           {i: r for i, r in enumerate(pos_results)})

    # Step 6: Save case file
    case_id   = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    case_data = {
        'case_id'         : case_id,
        'timestamp'       : datetime.now().isoformat(),
        'system'          : 'Digital Witness — YOLO26n + MobileNetV2 + POS',
        'video'           : {'path': str(video_path), 'duration': inf['duration'],
                              'fps': inf['fps'], 'frames': inf['total_frames']},
        'classification'  : {'label': inf['overall_class'],
                              'confidence': inf['overall_conf'],
                              'is_shoplifting': inf['is_shoplifting']},
        'behavior_events' : inf['behavior_events'],
        'person_summaries': inf['person_summaries'],
        'intent_score'    : intent,
        'bias_assessment' : bias,
        'alert'           : alert,
        'advisory_note'   : 'Advisory system only. Does not determine guilt.',
    }
    case_path = CASE_DIR / f'{case_id}.json'
    with open(case_path, 'w') as f:
        json.dump(case_data, f, indent=2)

    # Step 7: Generate POS integrity report
    pos_report, pos_report_path = generate_pos_report(
        inf, pos_results, case_id, OUTPUTS_DIR)

    # Step 8: Behaviour timeline plot
    if inf['behavior_events']:
        plot_behavior_timeline(
            inf['behavior_events'], inf['duration'],
            title=f'Behaviour Timeline — {Path(video_path).name}')

    print(f'\nCase file  : {case_path}')
    print(f'POS report : {pos_report_path}')

    return case_id, case_data, pos_report


# ---- Run it ----
# Replace with your actual test video path
# VIDEO_PATH = '/content/drive/MyDrive/DigitalWithness/Dataset/shoplifting/your_video.mp4'

# For a quick test with the suspicious mock session (timestamped to now):
# case_id, results, pos_report = analyze_video_with_pos(
#     VIDEO_PATH,
#     video_timestamp = datetime.now(),   # matches the suspicious session we added
#     frame_step      = 3,                # faster — every 3rd frame
# )

print("analyze_video_with_pos() ready.")
print()
print("To run:")
print("  case_id, results, pos_report = analyze_video_with_pos(")
print("      'path/to/video.mp4',")
print("      video_timestamp = datetime.now(),")
print("  )")
print()
print("When prompted:")
print("  - You will see what YOLO detected for each person")
print("  - You will see the matched POS transaction (if found)")
print("  - Enter the billed count to confirm or override")
print("  - System will flag any mismatch")
